In [26]:
# Imports

import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from PIL import Image
import kagglehub

In [27]:
#Dataset Download
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")
print("path",path)

Using Colab cache for faster access to the 'plantvillage' dataset.
path /kaggle/input/plantvillage


In [28]:
# Setup

torch.manual_seed(42)
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}")

 Device: cuda


In [29]:
# Defining Path

TRAIN_PATH = os.path.join(path, "PlantVillage","train")
VAL_PATH = os.path.join(path, "PlantVillage","val")

In [30]:
# Transformation

transform = transforms.Compose(
    [
        transforms.Resize( (128,128) ),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ]
)

In [31]:
# Custom Dataset

class MultiClassClassfication(Dataset):
    def __init__(self, root_dir, transform=None):
        super().__init__()

        self.samples = []
        self.transform = transform

        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(self.classes)}

        for class_name in self.classes:
            class_path = os.path.join(root_dir, class_name)

            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)

                if os.path.isfile(img_path):
                    label = self.class_to_idx[class_name]
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [32]:
# Loading full dataset

train_dataset_full = MultiClassClassfication(TRAIN_PATH, transform)
test_dataset_full = MultiClassClassfication(VAL_PATH, transform)
num_classes = len(train_dataset_full.classes)

print("Number of classes:", num_classes)
print("Full train size:", len(train_dataset_full))
print("Full test size:", len(test_dataset_full))

Number of classes: 38
Full train size: 43444
Full test size: 10861


In [33]:
# Creating Train and Test Dataset with Random Split

train_target_size = min(5000, len(train_dataset_full))
train_remainder = len(train_dataset_full) - train_target_size

train_dataset, _ = random_split(
    train_dataset_full,
    [train_target_size, train_remainder],
    generator=torch.Generator().manual_seed(42)
)

test_target_size = min(2000, len(test_dataset_full))
test_remainder = len(test_dataset_full) - test_target_size

test_dataset, _ = random_split(
    test_dataset_full,
    [test_target_size, test_remainder],
    generator=torch.Generator().manual_seed(42)
)

print(f"Successfully split datasets using PyTorch random_split!")
print(f"Selected Train Subset Size: {len(train_dataset)}")
print(f"Selected Test Subset Size: {len(test_dataset)}")

Successfully split datasets using PyTorch random_split!
Selected Train Subset Size: 5000
Selected Test Subset Size: 2000


In [34]:
# Data Loader

pin = True if device.type == 'cuda' else False
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=pin)

In [35]:
#CNN Model

class MyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2),

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(256, 256, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.MaxPool2d(2)
        )

        # Classifier part
        # Calculation: If input was 128x128, after 4 pools it is 8x8.
        # Feature map size = filters * H * W = 256 * 8 * 8
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 256), # Adjusted for 4th block and 256 filters
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [36]:
# Load model to GPU

model = MyCNN(num_classes=num_classes).to(device)

In [37]:
print(f"Model initialized with {num_classes} classes and 4 convolutional blocks.")

Model initialized with 38 classes and 4 convolutional blocks.


In [38]:
# Training Setup

learning_rate = 0.001
epochs = 15
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [39]:
# Training Loop

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/15, Loss: 2.8505
Epoch 2/15, Loss: 2.2013
Epoch 3/15, Loss: 1.8741
Epoch 4/15, Loss: 1.7001
Epoch 5/15, Loss: 1.5641
Epoch 6/15, Loss: 1.3730
Epoch 7/15, Loss: 1.2766
Epoch 8/15, Loss: 1.2216
Epoch 9/15, Loss: 1.1258
Epoch 10/15, Loss: 1.0736
Epoch 11/15, Loss: 1.0517
Epoch 12/15, Loss: 0.9490
Epoch 13/15, Loss: 0.8907
Epoch 14/15, Loss: 0.8547
Epoch 15/15, Loss: 0.8064


In [40]:
# Evaluation

def evaluate(loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for batch_features, batch_labels in loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            _, predicted = torch.max(outputs, 1)
            total += batch_labels.size(0)
            correct += (predicted == batch_labels).sum().item()
    return correct / total

In [41]:
train_acc = evaluate(train_loader)
print("Train Accuracy:", train_acc)

Train Accuracy: 0.8714


In [42]:
test_acc = evaluate(test_loader)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.775


In [43]:
torch.save(model.state_dict(), 'plantvillage_cnn.pth')

In [44]:
with open('classes.json', 'w') as f:
    json.dump(train_dataset_full.classes, f)